DuckDB load

In [ ]:
import os
import pandas as pd
import duckdb

# Ganti dengan path folder utama Anda tempat folder train berada
base_dir = r"D:\Downloads\BDC2026\train"
kategori = ["0_Recyclable", "1_Electronic", "2_Organic"]

dataset_info = []

for label in kategori:
    folder_path = os.path.join(base_dir, label)
    # Pastikan folder ada sebelum membaca isinya
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                dataset_info.append({
                    "filepath": os.path.join(folder_path, filename),
                    "label": label
                })

# Ubah list dictionary menjadi Pandas DataFrame
df_images = pd.DataFrame(dataset_info)
print(df_images)

# Membuat file database lokal bernama 'dataset_ku.duckdb'
# (Gunakan database=':memory:' jika hanya ingin berjalan di RAM sementara)
con = duckdb.connect(database='dataset_ku.duckdb')

# Membuat tabel 'image_metadata' di DuckDB dari DataFrame Pandas
con.execute("CREATE TABLE IF NOT EXISTS image_metadata AS SELECT * FROM df_images")

print("Data berhasil dimasukkan ke DuckDB!")

query_distribusi = """
SELECT label, COUNT(*) as jumlah_gambar 
FROM image_metadata 
GROUP BY label
ORDER BY label;
"""
print(con.execute(query_distribusi).df())

CNN Common

In [ ]:
"""
Shared helpers untuk CNN progression (dimodifikasi untuk lokal & struktur folder).
"""
from __future__ import annotations

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress TF info logs
os.environ.setdefault("PYTHONHASHSEED", "0")  # freeze hash for reproducibility
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # don't allocate all GPU memory at once

import gc
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report


# === MEMORY MONITORING ===
def get_memory_usage():
    """Return current memory usage in MB (works on Linux/Kaggle)."""
    try:
        import psutil
        process = psutil.Process(os.getpid())
        mem = process.memory_info().rss / (1024 * 1024)
        return mem
    except ImportError:
        return 0.0

def print_memory_usage(label: str = ""):
    """Print current memory usage with label."""
    mem = get_memory_usage()
    if mem > 0:
        print(f"  [MEMORY] {label}: {mem:.1f} MB")

def cleanup_memory():
    """Force garbage collection to free unused memory."""
    gc.collect()
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass

# === PATH LAYOUT (lokal) ===
BASE = Path(r"D:\Downloads\BDC2026")
DATASET = Path(r"D:\Downloads\BDC2026")

TRAIN_DIR = DATASET / "train"
TEST_DIR = DATASET / "test"

IMG_SIZE = 128
SEED = 42
tf.keras.utils.set_random_seed(SEED)


# === DATA LOADING ===

def _load_image(path: Path, img_size: int) -> np.ndarray:
    """Load 1 gambar, resize, normalize ke [0,1]. Return shape (img_size, img_size, 3)."""
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"gambar tidak bisa dibaca: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    return img / 255.0


def load_data_from_folders(data_dir: Path, img_size: int, has_labels: bool = True):
    """
    Load gambar + label dari struktur folder.
    Path gambar = data_dir / label / filename.
    Returns (X, y_or_None, ids, classes, class_to_idx).
    """
    print_memory_usage(f"Before loading data from {data_dir}")
    
    images = []
    labels = []
    ids = []
    classes = []
    class_to_idx = {}

    if has_labels:
        # Dapatkan daftar folder (kelas) dan buat mapping
        classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
        class_to_idx = {c: i for i, c in enumerate(classes)}
        
        count = 0
        for label_name in classes:
            label_dir = data_dir / label_name
            for img_path in label_dir.glob("*.*"): # Mengambil semua file (jpg, png, dll)
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    try:
                        img = _load_image(img_path, img_size)
                        images.append(img)
                        labels.append(class_to_idx[label_name])
                        ids.append(img_path.name)
                        count += 1
                        if count % 100 == 0:
                            print_memory_usage(f"Loaded {count} images")
                    except Exception as e:
                        print(f"Warning: Gagal memuat {img_path}: {e}")
    else:
        # Jika tidak ada label (misal data test dalam 1 folder tanpa subfolder)
        count = 0
        for img_path in data_dir.glob("*.*"):
             if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                try:
                    img = _load_image(img_path, img_size)
                    images.append(img)
                    ids.append(img_path.name)
                    count += 1
                    if count % 100 == 0:
                        print_memory_usage(f"Loaded {count} images")
                except Exception as e:
                    print(f"Warning: Gagal memuat {img_path}: {e}")

    # Konversi list ke numpy array
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32) if has_labels else None

    print_memory_usage(f"After loading data from {data_dir}")
    cleanup_memory()
    return X, y, ids, classes, class_to_idx


def load_train_data(img_size: int = IMG_SIZE):
    """Load train data. Returns (X, y, classes, class_to_idx)."""
    X, y, _, classes, class_to_idx = load_data_from_folders(TRAIN_DIR, img_size, has_labels=True)
    return X, y, classes, class_to_idx


def load_test_public(img_size: int = IMG_SIZE):
    """Load test data. Returns (X, ids)."""
    # Pastikan folder TEST_DIR ada. Jika test data juga dalam subfolder,
    # panggil dengan has_labels=True dan modifikasi kembaliannya.
    # Asumsi test data ada di dalam 1 folder tanpa subfolder label:
    if TEST_DIR.exists():
        X, _, ids, _, _ = load_data_from_folders(TEST_DIR, img_size, has_labels=False)
        return X, ids
    else:
        print("Warning: Folder test tidak ditemukan.")
        return None, None


# === SUBMISSION & GRADING ===
# (Fungsi grading dipertahankan, tapi akan gagal jika solution.csv tidak ada. 
# Kamu bisa menghapusnya jika tidak dibutuhkan lagi.)

def save_submission(ids: list[str], pred_labels: list[str], out_path: Path | None = None):
    if out_path is None:
        out_path = BASE / "submission.csv"
    out_df = pd.DataFrame({
        "id": ids,
        "label": pred_labels,
    })
    out_df.to_csv(out_path, index=False)
    print(f"  [SAVED] {out_path} ({len(pred_labels)} predictions)")
    return out_path


# === UTILITAS ===

def class_weights_from_labels(y: np.ndarray, num_classes: int) -> dict:
    from sklearn.utils.class_weight import compute_class_weight
    weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=y)
    return {i: float(w) for i, w in enumerate(weights)}

Training

In [ ]:
import os
import gc
import itertools
import duckdb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from PIL import Image
from tqdm import tqdm

# ==========================================
# === CONFIGURATION (CHAMPION EDITION v2) ===
# ==========================================
BASE = Path(r"D:\Downloads\BDC2026")
TEST_DIR = Path(r"D:\Downloads\BDC2026\test")
SAMPLE_SUBMISSION_PATH = Path(r"D:\Downloads\BDC2026\submission.csv")
DUCKDB_PATH = "dataset_ku.duckdb"

BATCH_SIZE = 16
NUM_EPOCHS = 10
K_FOLDS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS_TO_TRAIN = ["vit_b", "swin_b", "efficientnet_v2_l"]

# Berapa banyak "block" terakhir yang di-unfreeze untuk fine-tuning parsial.
# Head selalu trainable. Sisanya tetap frozen (hemat compute, kurangi overfit).
UNFREEZE_BLOCKS = {"vit_b": 2, "swin_b": 1, "efficientnet_v2_l": 2}

# Discriminative learning rate: backbone yang di-unfreeze dilatih jauh lebih pelan
# daripada head, supaya representasi pretrained tidak rusak (catastrophic forgetting).
HEAD_LR = {"vit_b": 1e-3, "swin_b": 2e-4, "efficientnet_v2_l": 5e-4}
BACKBONE_LR_FACTOR = 0.1  # backbone LR = HEAD_LR * factor

LABEL_SMOOTHING = 0.1
USE_TTA = True  # horizontal-flip TTA saat prediksi test

torch.manual_seed(42)
np.random.seed(42)


# ==========================================
# === DATASET CLASSES (RESOLUSI DINAMIS) ===
# ==========================================
class TrainValDataset(Dataset):
    def __init__(self, dataframe, label_mapping, img_size, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_mapping = label_mapping
        self.img_size = img_size

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'filepath']
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (self.img_size, self.img_size))
        if self.transform:
            image = self.transform(image)
        return image, self.label_mapping[self.dataframe.loc[idx, 'label']]


class TestDataset(Dataset):
    def __init__(self, dataframe, test_dir, img_size, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.test_dir = Path(test_dir)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_id = self.dataframe.loc[idx, 'id']
        img_name = f"{img_id}.jpg" if not str(img_id).endswith('.jpg') else str(img_id)
        img_path = self.test_dir / img_name
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (self.img_size, self.img_size))
        if self.transform:
            image = self.transform(image)
        return image, img_id


def get_transforms(img_size: int, train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])


# ==========================================
# === MODEL BUILDER (DENGAN PARTIAL UNFREEZE) ===
# ==========================================
def build_model(model_name: str, num_classes: int):
    n_unfreeze = UNFREEZE_BLOCKS[model_name]

    if model_name == "vit_b":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_SWAG_LINEAR_V1)
        for p in model.parameters():
            p.requires_grad = False
        for layer in model.encoder.layers[-n_unfreeze:]:
            for p in layer.parameters():
                p.requires_grad = True
        model.heads = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(model.heads.head.in_features, num_classes)
        )
        head_params = list(model.heads.parameters())
        backbone_params = [p for n, p in model.named_parameters()
                            if p.requires_grad and not n.startswith("heads")]

    elif model_name == "swin_b":
        model = models.swin_b(weights=models.Swin_B_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        for stage in model.features[-n_unfreeze:]:
            for p in stage.parameters():
                p.requires_grad = True
        model.head = nn.Linear(model.head.in_features, num_classes)
        head_params = list(model.head.parameters())
        backbone_params = [p for n, p in model.named_parameters()
                            if p.requires_grad and not n.startswith("head")]

    elif model_name == "efficientnet_v2_l":
        model = models.efficientnet_v2_l(weights=models.EfficientNet_V2_L_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        for stage in model.features[-n_unfreeze:]:
            for p in stage.parameters():
                p.requires_grad = True
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        head_params = list(model.classifier.parameters())
        backbone_params = [p for n, p in model.named_parameters()
                            if p.requires_grad and not n.startswith("classifier")]
    else:
        raise ValueError(f"Model tidak dikenal: {model_name}")

    return model.to(DEVICE), head_params, backbone_params


def build_optimizer(model_name, head_params, backbone_params):
    head_lr = HEAD_LR[model_name]
    backbone_lr = head_lr * BACKBONE_LR_FACTOR
    param_groups = [{"params": head_params, "lr": head_lr}]
    if len(backbone_params) > 0:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})

    if model_name == "vit_b":
        optimizer = torch.optim.Adam(param_groups)
    else:
        optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    return optimizer, scheduler


def compute_class_weights(df, label_mapping, num_classes):
    counts = np.zeros(num_classes)
    for label, idx in label_mapping.items():
        counts[idx] = (df['label'] == label).sum()
    weights = (1.0 / counts) * (len(df) / num_classes)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(inputs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


@torch.no_grad()
def predict_probs(model, loader, tta=False):
    """Prediksi softmax probs. Kalau tta=True, rata-ratakan dengan hasil horizontal-flip."""
    model.eval()
    all_probs, all_ids = [], []
    for inputs, batch_ids in loader:
        inputs = inputs.to(DEVICE)
        with torch.amp.autocast('cuda'):
            probs = F.softmax(model(inputs), dim=1)
            if tta:
                flipped = torch.flip(inputs, dims=[3])
                probs_flip = F.softmax(model(flipped), dim=1)
                probs = (probs + probs_flip) / 2.0
        all_probs.append(probs.float().cpu().numpy())
        all_ids.extend(batch_ids)
    return np.concatenate(all_probs, axis=0), all_ids


def search_best_weights(oof_probs_dict, y_true, step=0.05):
    """Grid search bobot ensemble (kelipatan `step`, jumlah=1) yang maksimalkan macro-F1 di OOF."""
    names = list(oof_probs_dict.keys())
    best_f1, best_weights = -1, None
    grid = np.arange(0, 1 + 1e-9, step)
    for w1 in grid:
        for w2 in grid:
            w3 = 1 - w1 - w2
            if w3 < -1e-9 or w3 > 1 + 1e-9:
                continue
            w3 = max(0, round(w3, 4))
            combo = (oof_probs_dict[names[0]] * w1 +
                     oof_probs_dict[names[1]] * w2 +
                     oof_probs_dict[names[2]] * w3)
            preds = np.argmax(combo, axis=1)
            f1 = f1_score(y_true, preds, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                best_weights = {names[0]: w1, names[1]: w2, names[2]: w3}
    return best_weights, best_f1


# ==========================================
# === MAIN EXECUTOR ===
# ==========================================
def main():
    print("=" * 64)
    print("TRAINING v2: partial fine-tune + class-weighted loss + OOF eval + TTA")
    print("=" * 64)

    con = duckdb.connect(DUCKDB_PATH)
    df = con.execute("SELECT filepath, label FROM image_metadata").df()
    classes = sorted(df['label'].unique().tolist())
    num_classes = len(classes)
    label_mapping = {c: i for i, c in enumerate(classes)}
    y_true_full = df['label'].map(label_mapping).values

    test_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    class_weights = compute_class_weights(df, label_mapping, num_classes)

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    scaler = torch.amp.GradScaler('cuda')

    all_models_test_probs = {m: np.zeros((len(test_df), num_classes)) for m in MODELS_TO_TRAIN}
    all_models_oof_probs = {m: np.zeros((len(df), num_classes)) for m in MODELS_TO_TRAIN}
    final_test_ids = []

    # --- PELATIHAN MODEL ---
    for current_model_name in MODELS_TO_TRAIN:
        current_img_size = 224 if current_model_name == "vit_b" else 256
        print(f"\n🚀 MELATIH: {current_model_name.upper()} (Resolusi: {current_img_size}x{current_img_size}, "
              f"unfreeze last {UNFREEZE_BLOCKS[current_model_name]} block)")

        test_dataset = TestDataset(test_df, TEST_DIR, current_img_size,
                                    transform=get_transforms(current_img_size, train=False))
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

        for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
            print(f"  ---> FOLD {fold + 1}/{K_FOLDS}")

            train_dataset = TrainValDataset(df.iloc[train_idx], label_mapping, current_img_size,
                                             transform=get_transforms(current_img_size, True))
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

            val_dataset = TrainValDataset(df.iloc[val_idx], label_mapping, current_img_size,
                                           transform=get_transforms(current_img_size, False))
            # dibungkus supaya cocok dengan predict_probs (butuh id, tapi kita pakai index asli)
            val_loader = DataLoader(
                list(zip([val_dataset[i][0] for i in range(len(val_dataset))], val_idx.tolist())),
                batch_size=BATCH_SIZE * 2, shuffle=False
            )

            model, head_params, backbone_params = build_model(current_model_name, num_classes)
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
            optimizer, scheduler = build_optimizer(current_model_name, head_params, backbone_params)

            for epoch in range(NUM_EPOCHS):
                train_one_epoch(model, train_loader, criterion, optimizer, scaler)
                if scheduler:
                    scheduler.step()
                if (epoch + 1) % 5 == 0:
                    print(f"       Epoch {epoch + 1}/{NUM_EPOCHS} selesai.")

            # --- OOF prediction (untuk evaluasi macro-F1 & pencarian bobot ensemble) ---
            val_probs, val_orig_idx = predict_probs(model, val_loader, tta=False)
            all_models_oof_probs[current_model_name][val_orig_idx] = val_probs
            fold_f1 = f1_score(y_true_full[val_orig_idx], np.argmax(val_probs, axis=1), average='macro')
            print(f"       Fold {fold + 1} macro-F1 (val): {fold_f1:.4f}")

            # --- Test prediction (dengan TTA) ---
            test_probs, test_ids = predict_probs(model, test_loader, tta=USE_TTA)
            all_models_test_probs[current_model_name] += test_probs / K_FOLDS
            if not final_test_ids:
                final_test_ids = test_ids

            del model, optimizer, train_loader, val_loader
            torch.cuda.empty_cache()
            gc.collect()

        oof_preds = np.argmax(all_models_oof_probs[current_model_name], axis=1)
        model_f1 = f1_score(y_true_full, oof_preds, average='macro')
        print(f"\n📊 {current_model_name.upper()} — OOF macro-F1 keseluruhan: {model_f1:.4f}")
        print(classification_report(y_true_full, oof_preds, target_names=classes, zero_division=0))

    # --- CARI BOBOT ENSEMBLE TERBAIK BERDASARKAN OOF (bukan tebakan manual) ---
    print("\n" + "=" * 64)
    print("MENCARI BOBOT ENSEMBLE TERBAIK VIA OOF GRID SEARCH...")
    print("=" * 64)
    best_weights, best_oof_f1 = search_best_weights(all_models_oof_probs, y_true_full, step=0.05)
    print(f"Bobot terbaik: {best_weights}")
    print(f"OOF macro-F1 dengan bobot ini: {best_oof_f1:.4f}")

    # --- APLIKASIKAN BOBOT TERBAIK KE TEST SET ---
    final_test_probs = sum(all_models_test_probs[m] * best_weights[m] for m in MODELS_TO_TRAIN)
    final_test_preds = np.argmax(final_test_probs, axis=1)

    clean_ids = [int(x.item()) if torch.is_tensor(x) else x for x in final_test_ids]
    submission_df = pd.DataFrame({'id': clean_ids, 'predicted': final_test_preds})
    submission_df.to_csv(BASE / "submission.csv", index=False)

    print(f"\nSELESAI! submission.csv dibuat dengan estimasi OOF macro-F1 ≈ {best_oof_f1:.4f}")


if __name__ == "__main__":
    main()